In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
documents = [
    "Eating fruits daily improves immunity.",
    "Drinking enough water keeps the body hydrated.",
    "Regular exercise strengthens the heart.",
    "Green vegetables are rich in vitamins and minerals.",
    "Protein helps in muscle growth and repair.",
    "Excess sugar consumption can lead to diabetes.",
    "Balanced diet is essential for overall health.",
    "Sleep is important for mental well-being.",
    "Whole grains support digestive health.",
    "Healthy fats like nuts and seeds are good for the brain."
]


In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert Sentences into Embeddings
# Each sentence becomes a numerical vector representation
embeddings = model.encode(documents)


# Save Embeddings Locally
np.save("../artifacts/health_embeddings.npy", embeddings)

print("Embeddings generated and stored successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings generated and stored successfully!


In [9]:
# Load Stored Embeddings
stored_embeddings = np.load("../artifacts/health_embeddings.npy")

#query's

#query = "Why is drinking water important?"
#query = "How can I make my heart stronger?"
#query = "Which foods are good for brain health?"
query = "What are the risks of eating too much sugar?"
#query = "Why is sleep important for mental health?"
#query = "Why is a balanced diet necessary?"
#query = "What foods contain important vitamins?"

# Convert query into embedding
query_embedding = model.encode([query])
print("Query Embedding completed successfully!")

Query Embedding completed successfully!


In [6]:
# Compute Cosine Similarity

similarities = cosine_similarity(query_embedding, stored_embeddings)

# Identify Most Relevant Sentence
best_match_index = np.argmax(similarities)
print("Best Match Index:", best_match_index)

Best Match Index: 3


In [7]:
# Display Result

print("Query:", query)
print("Most Relevant Sentence:", documents[best_match_index])
print("Similarity Score:", similarities[0][best_match_index])

Query: What foods contain important vitamins?
Most Relevant Sentence: Green vegetables are rich in vitamins and minerals.
Similarity Score: 0.6054182


how to push code in pinecone

In [8]:
# ---------------------------------------
# Store Existing Embeddings in Pinecone
# ---------------------------------------
from pinecone import Pinecone, ServerlessSpec
import time

# 1️⃣ Initialize Pinecone
pc = Pinecone(api_key="pcsk_UrePP_56vSCDDph7vqheXLnBZzXn8gRVSmZbwpU8YWA559S8UXDsjFGv7tQ9UCofYD7hv")  # Replace with your real key

index_name = "health-demo"

# 2️⃣ Create Index (Only if not exists)
if index_name not in [i["name"] for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=embeddings.shape[1],  # Automatically get dimension (384)
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    time.sleep(10)

# 3️⃣ Connect to Index
index = pc.Index(index_name)

# 4️⃣ Prepare Vectors (Using Already Created embeddings)
vectors = []

for i, doc in enumerate(documents):
    vectors.append(
        (
            str(i),                     # Unique ID
            embeddings[i].tolist(),     # Existing embedding
            {"text": doc}               # Metadata
        )
    )

# 5️⃣ Upload to Pinecone
index.upsert(vectors=vectors)

print("Embeddings successfully stored in Pinecone!")

Embeddings successfully stored in Pinecone!
